# Results for new splits

Results from `scripts/run_chelsa_monthly.sh` (linear probe per model + IDW baseline).

In [ ]:
import ast
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.environ.setdefault("RESULTS_HOME", "/projects/bgtj/t_satclip/results/new_splits")
DATASET = "chelsa_no_antarctica"
DATASET_TITLE = "CHELSA monthly (no Antarctica)"

CSV = Path(os.environ["RESULTS_HOME"]) / "chelsa_monthly.csv"
OUT_DIR = CSV.parent / "plots"
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
df = pd.read_csv(CSV)
df = df[df["dataset"] == DATASET].copy()
assert len(df), f"no rows for dataset={DATASET!r}; found {sorted(pd.read_csv(CSV)['dataset'].unique())}"

# Rerunning the script with --resume can append the same result again; keep only the most recent row.
key = ["task", "method", "model_name", "split", "inner_split", "split_params",
       "temporal_embedding_aggregation"]
df = df.drop_duplicates(subset=key, keep="last")

# Split label, e.g. "random/random (year,1)" or "temporal_checkerboard/random (season,1)"
params = df["split_params"].map(ast.literal_eval)
df["split_label"] = (
    df["split"] + "/" + df["inner_split"]
    + " (" + params.map(lambda p: f"{p.get('time_unit')},{p.get('time_unit_size')}") + ")"
)

# Readable model label; IDW runs without a model
df["model"] = np.where(
    df["method"] == "idw",
    "IDW",
    df["model_name"].fillna("").astype(str).str.split("/").str[-1],
)
df["std"] = df["metric_value"] - df["ci_lower"]

# Keep the run order from the shell script, with IDW last as the reference.
MODEL_ORDER = [
    "climplicit",
    "t_satclip_toy_y_300k_satcliploss",
    "t_satclip_toy_y_300k_softloss",
    "t_satclip_toy_y_100k",
    "t_satclip_toy_y_150k",
    "t_satclip_toy_y_200k",
    "t_satclip_toy_y_250k",
    "t_satclip_toy_y_300k",
    "t_satclip_y_300k",
    "t_satclip_toy_300k",
    "gtloc",
    "IDW",
]
models = [m for m in MODEL_ORDER if m in set(df["model"])] + sorted(
    set(df["model"]) - set(MODEL_ORDER)
)
splits = sorted(df["split_label"].unique())
metric = df["metric_name"].iloc[0]
df.head()

## Mean across tasks, per model and split

In [ ]:
summary = (
    df.groupby(["model", "split_label"])
    .agg(mean=("metric_value", "mean"), std=("metric_value", "std"), n_tasks=("task", "nunique"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(models) * len(splits)), 5))
width = 0.8 / len(splits)
x = np.arange(len(models))
for i, split in enumerate(splits):
    s = summary[summary["split_label"] == split].set_index("model").reindex(models)
    ax.bar(x + (i - (len(splits) - 1) / 2) * width, s["mean"], width,
           yerr=s["std"], capsize=2, label=split)
ax.set_xticks(x, models, rotation=45, ha="right")
ax.set_ylabel(f"Mean {metric} across tasks (± std across tasks)")
ax.set_title(f"{DATASET_TITLE} — linear probe vs IDW")
ax.axhline(0, color="black", linewidth=0.8)
ax.legend(title="outer/inner split (time unit)", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "chelsa_monthly_summary.png", dpi=200)
plt.show()

## Per-task results

In [ ]:
tasks = sorted(df["task"].unique())
fig, axes = plt.subplots(1, len(splits),
                         figsize=(1 + 0.6 * len(tasks) * len(splits), 0.45 * len(models) + 2),
                         sharey=True, squeeze=False)
vmin, vmax = df["metric_value"].min(), df["metric_value"].max()
for ax, split in zip(axes[0], splits):
    grid = (df[df["split_label"] == split]
            .pivot_table(index="model", columns="task", values="metric_value")
            .reindex(index=models, columns=tasks))
    im = ax.imshow(grid.to_numpy(), aspect="auto", cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(tasks)), tasks, rotation=60, ha="right")
    ax.set_yticks(range(len(models)), models)
    ax.set_title(split.replace(" (", "\n("))
    for (r, c), v in np.ndenumerate(grid.to_numpy()):
        if not np.isnan(v):
            ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if v < (vmin + vmax) / 2 else "black")
fig.colorbar(im, ax=axes[0].tolist(), label=metric, shrink=0.8)
fig.suptitle(f"{DATASET_TITLE} — per-task results")
fig.savefig(OUT_DIR / "chelsa_monthly_per_task.png", dpi=200, bbox_inches="tight")
plt.show()

## Summary table

In [ ]:
summary.pivot(index="model", columns="split_label", values="mean").reindex(models).round(3)